Importar bibliotecas y preparación de datos

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from pathlib import Path

In [3]:
ruta_carpeta = Path(r'/content/drive/MyDrive/Colab Notebooks/redes neuronales/dataset violencia intrafamiliar')

archivos_excel = list(ruta_carpeta.rglob('*.xlsx'))

In [4]:
lista_dfs = []

for archivo in archivos_excel:
    try:
        df = pd.read_excel(archivo)
        lista_dfs.append(df)
        print(f"Cargado: {archivo.name}")
    except Exception as e:
        print(f"Error cargando {archivo.name}: {e}")

if lista_dfs:
    datos = pd.concat(lista_dfs, ignore_index=True)
    print("\n¡Carga completa!")
else:
    print("No se encontraron archivos Excel.")

Cargado: base-de-datos-violencia-intrafamiliar-ano-2024_v3.xlsx
Cargado: diccionario-de-variables-violencia-intrafamiliar-2023.xlsx

¡Carga completa!


In [38]:
df_final = datos

In [39]:
df_final.head()

,HEC_DIA,HEC_MES,HEC_ANO,HEC_DEPTO,HEC_DEPTOMCPIO,HEC_TIPAGRE,NUMERO_BOLETA,DIA_EMISION,MES_EMISION,ANO_EMISION,...,ARTICULOTRAS3,ARTICULOTRAS4,MEDIDAS_SEGURIDAD,TIPO_MEDIDA,ORGANISMO_REMITE,ESTADÍSTICAS DE VIOLENCIA INTRAFAMILIAR,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,4.0,11.0,2024.0,1.0,110.0,1122.0,367.0,4.0,11.0,2024.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,24.0,3.0,2024.0,2.0,202.0,1222.0,5.0,25.0,3.0,2024.0,...,NaN,NaN,1.0,IJ,18.0,NaN,NaN,NaN,NaN,NaN
2,99.0,99.0,9999.0,1.0,101.0,1122.0,430.0,2.0,3.0,2024.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28.0,3.0,2024.0,2.0,202.0,1122.0,6.0,28.0,3.0,2024.0,...,NaN,NaN,1.0,AIJ,18.0,NaN,NaN,NaN,NaN,NaN
4,12.0,7.0,2024.0,7.0,706.0,2122.0,16.0,24.0,7.0,2024.0,...,NaN,NaN,1.0,IJ,18.0,NaN,NaN,NaN,NaN,NaN


In [40]:
df_final = df_final[["VIC_SEXO", "AGR_SEXO", "AGR_EDAD", "AGR_ALFAB", "AGR_TRABAJA", "AGR_EST_CIV", "AGR_DEDICA", "AGR_ESCOLARIDAD"]]

In [41]:
df_final = df_final.dropna()

In [42]:
df_final['feature'] = df_final['VIC_SEXO'].apply(lambda x: 1 if x == 1 else 0)

In [43]:
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

Definición de variables independientes y dependientes (a predecir); definición de conjunto de entreno y pruebas

In [44]:
x = df_final[[
    "AGR_SEXO", "AGR_EDAD", "AGR_ALFAB", "AGR_TRABAJA", "AGR_EST_CIV", "AGR_DEDICA", "AGR_ESCOLARIDAD"
]]
y = df_final['feature']

In [45]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

diseño y entreno de la red neuronal

In [46]:
model = Sequential()
model.add(Dense(8, input_dim=7, activation='relu'))
model.add(Dense(4, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [47]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [48]:
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_data=(x_test, y_test))

Epoch 1/5
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5693 - loss: 0.9295 - val_accuracy: 0.6459 - val_loss: 0.6635
Epoch 2/5
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6857 - loss: 0.6336 - val_accuracy: 0.7009 - val_loss: 0.6187
Epoch 3/5
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7125 - loss: 0.6034 - val_accuracy: 0.7306 - val_loss: 0.5947
Epoch 4/5
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7266 - loss: 0.5895 - val_accuracy: 0.7375 - val_loss: 0.5843
Epoch 5/5
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7344 - loss: 0.5726 - val_accuracy: 0.7311 - val_loss: 0.5764


In [49]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f'loss: {loss}, Accuracy: {accuracy}')

73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7187 - loss: 0.5783
loss: 0.576430082321167, Accuracy: 0.7310671210289001


predicción

In [50]:
persona = np.array([[
    2,
  35,
  1,
  2,
  2,
  4,
  32
]])
res = model.predict(persona)
print(res)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
[[0.3317499]]
